# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhruv6305/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- **Unit of analysis**: One row = one content item per client per day (`report_date` × `client_hash_id` × `content_hash_id`).
- **Table**: `fact_content_daily_performance` (we use the `month=2026-03` partition for development to avoid hitting the final test month).
- **Time window**: `2026-03-01` to `2026-03-31`.

In [15]:
import duckdb
import os

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except ImportError:
    token = os.environ.get("HF_TOKEN")

con = duckdb.connect()
if token:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

# Point to the 2026-03 partition on Hugging Face using DuckDB's remote Parquet reader
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=1)"

# Optional: test connection by reading metadata
# con.sql(f"SELECT COUNT(*) FROM {REL}").show()


## 2. Fields: feature / label / context / excluded

- **Feature**: `prev_impressions`, `prev_position`, `prev_sessions`, `prev_pageviews`, `prev_clicks` — these are lagged features from the previous day, knowable *before* predicting today's clicks.
- **Label / proxy**: `has_clicks_label` — binary indicator if `gsc_clicks > 0` on the current day.
- **Context**: `client_hash_id`, `content_hash_id`, `report_date` — for grouping and splits, never to be learned by the model.
- **Excluded**: Today's `gsc_impressions` and `gsc_clicks` (because they happen concurrently with the label and would cause leakage) and rows where `ga4_data_available IS NOT TRUE`.

In [16]:
context_cols = ['client_hash_id', 'content_hash_id', 'report_date']
feature_cols = [
    'prev_impressions',
    'prev_position',
    'prev_sessions',
    'prev_pageviews',
    'prev_clicks'
]
label_col = 'has_clicks_label'
excluded_cols = ['gsc_clicks', 'gsc_impressions']


## 3. Verify it with queries (grain, counts, missing values, windows)

**Three Facts:**
1. **Grain:** One row really is `report_date × client_hash_id × content_hash_id`. The GROUP BY count should return zero rows.
2. **Row count and date span:** We query exactly how many rows are in the `2026-03` slice.
3. **Availability:** We filter by `ga4_data_available IS TRUE` and check how many rows survive.

**Five Features Frame:**
1. `prev_impressions`: Knowable at the decision moment because yesterday's impressions are already recorded.
2. `prev_position`: Knowable at the decision moment because we use the rank from the day prior.
3. `prev_sessions`: Knowable at the decision moment because past visits have concluded.
4. `prev_pageviews`: Knowable at the decision moment because yesterday's pageviews are fixed.
5. `prev_clicks`: Knowable at the decision moment because yesterday's clicks happened in the past.

**The Trap:**
We deliberately include today's `gsc_clicks` as a feature (a direct leak of the label `gsc_clicks > 0`). We watch the ROC AUC jump to 1.0 (Leakage), then we remove it and get the honest score based purely on the previous day's metrics.

In [17]:
# 1. Grain
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
display(grain_check)

# 2. Counts and Window
counts_check = con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM {REL}
""").df()
print("\nCounts and Window:")
display(counts_check)

# 3. Availability - filter with IS TRUE
availability_check = con.sql(f"""
    SELECT COUNT(*) as surviving_rows
    FROM {REL}
    WHERE ga4_data_available IS TRUE
""").df()
print("\nAvailability (ga4_data_available IS TRUE):")
display(availability_check)

# === THE TRAP (Leakage Lesson) ===
df = con.sql(f"""
    WITH daily_data AS (
        SELECT 
            report_date, client_hash_id, content_hash_id,
            COALESCE(gsc_clicks, 0) as gsc_clicks,
            COALESCE(gsc_impressions, 0) as gsc_impressions,
            COALESCE(gsc_avg_position, 0) as gsc_avg_position,
            COALESCE(ga4_sessions, 0) as ga4_sessions,
            COALESCE(ga4_pageviews, 0) as ga4_pageviews
        FROM {REL}
        WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE
        LIMIT 200000
    )
    SELECT 
        CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END AS has_clicks_label,
        LAG(gsc_impressions) OVER w AS prev_impressions,
        LAG(gsc_avg_position) OVER w AS prev_position,
        LAG(ga4_sessions) OVER w AS prev_sessions,
        LAG(ga4_pageviews) OVER w AS prev_pageviews,
        LAG(gsc_clicks) OVER w AS prev_clicks,
        gsc_clicks AS TRAP_leakage_clicks
    FROM daily_data
    WINDOW w AS (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date)
""").df()

df = df.dropna()

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

y = df['has_clicks_label']

# WITH THE TRAP (Leakage)
X_leak = df[['prev_impressions', 'prev_position', 'prev_sessions', 'prev_pageviews', 'prev_clicks', 'TRAP_leakage_clicks']]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_l_scaled = scaler.fit_transform(X_train_l)
X_test_l_scaled = scaler.transform(X_test_l)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_l_scaled, y_train_l)
preds_leak = model.predict_proba(X_test_l_scaled)[:, 1]
print(f"\nROC AUC WITH LEAKAGE (TRAP_leakage_clicks included): {roc_auc_score(y_test_l, preds_leak):.4f}")

# WITHOUT THE TRAP (Honest)
X_honest = df[['prev_impressions', 'prev_position', 'prev_sessions', 'prev_pageviews', 'prev_clicks']]
X_train_h, X_test_h, _, _ = train_test_split(X_honest, y, test_size=0.2, random_state=42)

X_train_h_scaled = scaler.fit_transform(X_train_h)
X_test_h_scaled = scaler.transform(X_test_h)

model.fit(X_train_h_scaled, y_train_l)
preds_honest = model.predict_proba(X_test_h_scaled)[:, 1]
print(f"ROC AUC HONEST (TRAP_leakage_clicks removed): {roc_auc_score(y_test_l, preds_honest):.4f}")


Grain violations (should be empty):


,report_date,client_hash_id,content_hash_id,c



Counts and Window:


,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31



Availability (ga4_data_available IS TRUE):


,surviving_rows
0,413966



ROC AUC WITH LEAKAGE (TRAP_leakage_clicks included): 1.0000
ROC AUC HONEST (TRAP_leakage_clicks removed): 0.7877


## 4. Data limits

**Limitation:** Unbalanced history across clients.
Some clients connected GA4 much later than others, so their early rows are zero-filled with `ga4_data_available = FALSE`. We must explicitly filter these out using `IS TRUE` rather than treating them as days with zero engagement. 

In [18]:
start_dates = con.sql(f"""
    SELECT client_hash_id, MIN(report_date) as first_active_date
    FROM {REL}
    WHERE ga4_data_available IS TRUE
    GROUP BY client_hash_id
    ORDER BY first_active_date DESC
    LIMIT 5
""").df()
print("Clients connected GA4 at completely different times:")
display(start_dates)


Clients connected GA4 at completely different times:


,client_hash_id,first_active_date
0,client_810019792c9b8efc,2026-03-26
1,client_e00b29e582949543,2026-03-25
2,client_73cda7b4e4f265ea,2026-03-24
3,client_b77d0d5f08f05e64,2026-03-19
4,client_7eafe750768f0ac2,2026-03-14


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.